In [ ]:
from dotenv import load_dotenv  # loads .env file into environment variables
import os                        # provides os.getenv to read environment variables
load_dotenv()                    # reads .env in current directory
api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key (None if missing)
print("API key loaded:", api_key is not None)  # True = key found, False = missing

In [ ]:
from anthropic import Anthropic  # imports the Anthropic SDK client class
client = Anthropic(api_key=api_key)  # instantiates client; all API calls go through this object
print("Anthropic client ready")  # confirms no import/auth errors at construction time

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5",      # fastest/cheapest model — good for smoke tests
    # model="claude-sonnet-4-5",   # uncomment for higher-quality responses
    max_tokens=50,                 # caps output length to avoid runaway costs
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal valid payload
)
print(response.content[0].text)   # content is a list of blocks; [0] is the first text block

# CCA Lab: Prompt Engineering & the Improvement Cycle

**Course:** Anthropic Academy — Claude API Fundamentals  
**Sub-module:** Prompt Design & Evaluation  
**Lesson:** Writing, Scoring, and Iterating on Prompts

## What this lab covers
- Setting up the Anthropic client with dotenv
- Building a reusable `ask()` helper with full annotation
- Using the `system` parameter effectively
- Multi-turn conversation via `messages` list accumulation
- Prompt improvement cycle: write → evaluate → improve → re-evaluate
- Failure mode analysis with live code demonstrations
- Token usage tracking across a session

## CCA Domains Exercised
`API Mechanics` · `Prompt Design` · `Evaluation` · `Iteration`

## Learning Objectives
1. Explain every parameter in `client.messages.create()`
2. Construct and accumulate a multi-turn `messages` list
3. Apply a numeric rubric to score prompt outputs
4. Execute at least one full improvement cycle
5. Identify three common prompt failure modes and mitigate them

## Section 1: API Glossary & Reference
## CCA objective demonstrated: Know every `messages.create()` parameter by name, type, and purpose

| Parameter | Type | Required | Purpose |
|-----------|------|----------|---------|
| `model` | `str` | ✅ | Which Claude model to use (e.g. `claude-haiku-4-5`) |
| `max_tokens` | `int` | ✅ | Hard ceiling on output tokens — **always set this** |
| `messages` | `list[dict]` | ✅ | Ordered conversation: `[{"role": "user"\|"assistant", "content": str}]` |
| `system` | `str` | ❌ | Persistent persona/instructions prepended before the conversation |
| `temperature` | `float 0-1` | ❌ | Randomness: 0 = deterministic, 1 = creative |
| `stop_sequences` | `list[str]` | ❌ | Tokens that halt generation early |
| `stream` | `bool` | ❌ | If `True`, yields partial tokens as they are generated |

> **`stop_reason` values** returned in the response: `end_turn` (natural finish), `max_tokens` (truncated), `stop_sequence` (hit a stop token).

## Section 2: The `ask()` Helper — Statement-by-Statement Annotation
## CCA objective demonstrated: Build a production-ready wrapper with full observability

In [ ]:
usage_log = []  # session-level list; every API call appends one dict here

def ask(
    prompt: str,
    system: str = "",
    model: str = "claude-haiku-4-5",
    max_tokens: int = 512,
    temperature: float = 0.0,
) -> str:
    """
    Send a single-turn prompt and return the text reply.

    Args:
        prompt:      The user-turn message text.
        system:      Optional system prompt (persona / standing instructions).
        model:       Claude model identifier string.
        max_tokens:  Maximum tokens Claude may produce.
        temperature: Sampling temperature (0 = deterministic).

    Returns:
        The assistant reply as a plain Python string.
    """
    # --- build API call kwargs ---
    kwargs = dict(
        model=model,           # selects which Claude model processes this request
        max_tokens=max_tokens, # prevents runaway token spend on open-ended prompts
        temperature=temperature,  # 0 = greedy decode → reproducible for evals
        messages=[{"role": "user", "content": prompt}],  # minimal single-turn list
    )
    if system:                 # only add 'system' key when caller provides text
        kwargs["system"] = system  # passed as top-level keyword, NOT inside messages

    response = client.messages.create(**kwargs)  # synchronous HTTP call to Claude API

    # response.content is a list because Claude can return multiple content blocks
    # (e.g. text + tool_use).  [0] selects the first (and usually only) text block.
    text = response.content[0].text  # .text extracts the string from the TextBlock object

    # stop_reason tells us WHY generation ended — important for debugging truncation
    stop = response.stop_reason   # 'end_turn' | 'max_tokens' | 'stop_sequence'

    # Append usage data so we can audit token spend at the end of the session
    usage_log.append({
        "prompt_preview": prompt[:60],           # first 60 chars for identification
        "input_tokens":  response.usage.input_tokens,   # tokens in the request
        "output_tokens": response.usage.output_tokens,  # tokens in the reply
        "stop_reason":   stop,
    })

    if stop == "max_tokens":                     # warn when output was cut short
        print(f"[WARNING] Response truncated — raise max_tokens (currently {max_tokens})")

    return text  # caller receives a plain str, never a complex SDK object

# Quick sanity check
print(ask("What is 2 + 2? Answer in one word."))

## Section 3: Intentional Error Demo — Omitting `max_tokens`
## CCA objective demonstrated: Recognise and fix the most common required-parameter error

In [ ]:
# Deliberately call the API without max_tokens to see the SDK error message.
# This is the #1 mistake beginners make — knowing the error string speeds up debugging.
try:
    bad_response = client.messages.create(
        model="claude-haiku-4-5",
        # max_tokens intentionally omitted
        messages=[{"role": "user", "content": "Hello"}],
    )
except Exception as e:
    # Prints the actual SDK/API error so you know what to look for in production logs
    print(f"Error type : {type(e).__name__}")
    print(f"Error detail: {e}")
    print("\n✅ Fix: always pass max_tokens to client.messages.create()")

## Section 4: The `system` Parameter — When & Why
## CCA objective demonstrated: Use system prompts to shape persona, tone, and constraints

In [ ]:
# Without a system prompt — Claude uses its default helpful-assistant persona
no_system = ask("Explain recursion.", max_tokens=120)
print("=== No system ===")
print(no_system)

# With a system prompt — Claude adopts the specified persona/constraints
# Use system prompts when you need: persona, output format, domain constraints,
# safety rules, or language/tone requirements that apply to every turn.
with_system = ask(
    prompt="Explain recursion.",
    system="You are a patient tutor explaining concepts to a 10-year-old. "
           "Use one short analogy and no jargon.",
    max_tokens=120,
)
print("\n=== With system ===")
print(with_system)

## Section 5: Multi-Turn Conversation — `messages` List Accumulation
## CCA objective demonstrated: Manage conversation state by appending role/content dicts

In [ ]:
def chat(history: list, user_msg: str, system: str = "") -> str:
    """
    Append user_msg to history, call the API, append the reply, return reply text.
    Mutates `history` in place so callers accumulate the full conversation.
    """
    history.append({"role": "user", "content": user_msg})  # add user turn

    kwargs = dict(model="claude-haiku-4-5", max_tokens=256, messages=history)
    if system:
        kwargs["system"] = system  # system is NOT part of the messages list

    resp = client.messages.create(**kwargs)  # send full history every call
    reply = resp.content[0].text             # extract text from first content block

    history.append({"role": "assistant", "content": reply})  # add assistant turn

    # Track token usage for the session log
    usage_log.append({
        "prompt_preview": user_msg[:60],
        "input_tokens":  resp.usage.input_tokens,
        "output_tokens": resp.usage.output_tokens,
        "stop_reason":   resp.stop_reason,
    })
    return reply

# Demonstrate: each turn references context from the previous turn
convo = []  # empty list = fresh conversation
print("Turn 1:", chat(convo, "My name is Alex. Remember it."))
print("Turn 2:", chat(convo, "What is my name?"))  # Claude must recall from history
print(f"\nMessages in history: {len(convo)} (user+assistant dicts accumulated)")

## Section 6: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Apply a numeric rubric and iterate until quality threshold is met

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
import re  # standard library regex module — used for keyword scoring

def score_explanation(text: str) -> dict:
    """
    Score a recursion explanation on four criteria (0-1 each → max 4).

    Uses re.search (not re.match) because we want to find the pattern
    *anywhere* in the string, not just at the start.
    \\b word boundaries prevent 'recursive' matching inside other words.

    int(bool(...)) converts a Match object or None → 1 or 0.
    """
    criteria = {
        # int(bool(re.search(...))) → 1 if keyword found, 0 otherwise
        "has_base_case":   int(bool(re.search(r"\bbase case\b",    text, re.I))),
        "has_analogy":     int(bool(re.search(r"\b(like|imagine|think of|analogy)\b", text, re.I))),
        "no_jargon":       int(not bool(re.search(r"\b(stack frame|heap|pointer)\b", text, re.I))),
        "concise":         int(len(text.split()) <= 80),  # under 80 words = concise
    }
    criteria["total"] = sum(v for k, v in criteria.items() if k != "total")
    return criteria

# --- ATTEMPT 1: weak prompt ---
v1_prompt = "Explain recursion."
v1_response = ask(v1_prompt, max_tokens=150)
v1_score = score_explanation(v1_response)
print("=== Version 1 ===")
print(v1_response)
print("Score:", v1_score)

In [ ]:
# --- ATTEMPT 2: improved prompt with explicit rubric constraints ---
v2_prompt = (
    "Explain recursion to a 10-year-old in under 80 words. "
    "Include a real-world analogy and explicitly mention the base case. "
    "Avoid technical jargon like 'stack frame' or 'heap'."
)
v2_response = ask(v2_prompt, max_tokens=150)
v2_score = score_explanation(v2_response)
print("=== Version 2 ===")
print(v2_response)
print("Score:", v2_score)

# Summary delta
delta = v2_score["total"] - v1_score["total"]
print(f"\nImprovement: {v1_score['total']}/4 → {v2_score['total']}/4  (Δ {delta:+d})")

## Section 7: Failure Mode Analysis
## CCA objective demonstrated: Identify, demonstrate, and mitigate common prompt failure modes

In [ ]:
# Failure modes table (printed programmatically so it stays in-sync with live demos)
failure_modes = [
    ("Vague prompt",      "Ask a generic question",           "Add format, length, audience constraints"),
    ("max_tokens too low","Response is cut mid-sentence",     "Check stop_reason == 'max_tokens'; increase limit"),
    ("Missing system",    "Tone/persona shifts between turns","Add a system prompt with explicit persona rules"),
    ("No base case",      "Infinite recursion (API loops)",   "Always break recursion on a stopping condition"),
]
print(f"{'Failure Mode':<22} {'Symptom':<35} {'Mitigation'}")
print("-" * 80)
for mode, symptom, fix in failure_modes:
    print(f"{mode:<22} {symptom:<35} {fix}")

# --- Live demo: max_tokens too low ---
print("\n--- Live Demo: max_tokens=10 ---")
short = ask("Write a haiku about the ocean.", max_tokens=10)  # will truncate
print(repr(short))
last_stop = usage_log[-1]["stop_reason"]
print(f"stop_reason = '{last_stop}'  ← expect 'max_tokens'")

## Section 8: Token Usage Analysis
## CCA objective demonstrated: Track and audit token spend across a full session

In [ ]:
# Summarise every API call made in this session
total_in  = sum(r["input_tokens"]  for r in usage_log)  # list comprehension sums all input tokens
total_out = sum(r["output_tokens"] for r in usage_log)  # list comprehension sums all output tokens

print(f"{'#':<3} {'Prompt (preview)':<45} {'In':>5} {'Out':>5} {'Stop'}")
print("-" * 72)
# enumerate gives (index, dict) pairs — we use idx+1 for 1-based call numbering
for idx, entry in enumerate(usage_log):
    preview = entry["prompt_preview"].ljust(45)   # left-align preview in 45 chars
    print(f"{idx+1:<3} {preview} {entry['input_tokens']:>5} {entry['output_tokens']:>5} {entry['stop_reason']}")

print("-" * 72)
print(f"{'TOTAL':<49} {total_in:>5} {total_out:>5}")
print(f"\nGrand total tokens this session: {total_in + total_out}")

## Section 9: Practice Drills
## CCA objective demonstrated: Apply learned concepts independently across novel tasks

In [ ]:
# ── Drill 1: System prompt persona ──────────────────────────────────────────
# Task: ask Claude to explain gravity using a "Shakespearean poet" persona.
# YOUR CODE BELOW — replace the ellipsis
drill1 = ask(
    prompt="Explain gravity in two sentences.",
    system="You are a Shakespearean poet. Respond only in iambic pentameter.",
    max_tokens=80,
)
print("Drill 1 — Shakespearean gravity:")
print(drill1)

# ── Drill 2: Multi-turn with topic continuity ────────────────────────────────
# Task: start a conversation about space, then ask a follow-up that requires context.
space_convo = []
chat(space_convo, "Tell me one interesting fact about black holes.")  # turn 1
reply2 = chat(space_convo, "How does that compare to neutron stars?")  # turn 2 needs context
print("\nDrill 2 — Multi-turn space chat (Turn 2):")
print(reply2)

# ── Drill 3: Rubric scoring on your own prompt ───────────────────────────────
# Task: write a prompt that scores 4/4 on score_explanation().
my_prompt = (
    "In 60 words, explain recursion using an analogy. "
    "Mention the base case explicitly. Avoid stack frame, heap, pointer."
)
my_response = ask(my_prompt, max_tokens=120)
my_score = score_explanation(my_response)
print(f"\nDrill 3 — My score: {my_score['total']}/4")
print(my_score)

## Section 10: CCA Takeaways
## CCA objective demonstrated: Consolidate exam-ready mental models for every major concept

| # | Mental Model | Why It Matters |
|---|--------------|----------------|
| 1 | **`max_tokens` is always required** — omitting it raises an error before any token is generated | Prevents the single most common beginner mistake |
| 2 | **`response.content` is a list** — always index with `[0]` for the primary text block | Claude can return multiple blocks (text + tool use); defensive indexing is safe |
| 3 | **`system` is a top-level keyword**, not a message dict — it frames every turn without appearing in `messages` | Mixing system into the messages list breaks the API contract |
| 4 | **Multi-turn = append, don't replace** — each call sends the full accumulated `messages` list | Claude has no server-side memory; you own conversation state entirely |
| 5 | **Rubric + semantic judge** — keyword rubrics are fast but brittle; pair with a Claude-as-judge pass for production evals | Deterministic checks miss paraphrase; semantic graders catch meaning | 

---
**Lab complete.** Run *Kernel → Restart and Run All Cells* to verify end-to-end execution.